# Information for paper 1

## Mergging: old feature file with new one - Codrin "Centroid"(21/06/26)

In [ ]:
import pandas as pd

OLD_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_19_05_26.csv"
NEW_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_conf_0.5_centroid.csv"
OUT_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv"

print("Loading files...")
df_old = pd.read_csv(OLD_PATH)
df_new = pd.read_csv(NEW_PATH)

print(f"Old file : {df_old.shape[0]:,} rows x {df_old.shape[1]:,} columns")
print(f"New file : {df_new.shape[0]:,} rows x {df_new.shape[1]:,} columns")

# --- check image_name overlap ---
images_old = set(df_old["image_name"])
images_new = set(df_new["image_name"])
print(f"\nImages in old only : {len(images_old - images_new):,}")
print(f"Images in new only : {len(images_new - images_old):,}")
print(f"Images in both     : {len(images_old & images_new):,}")

# --- find duplicate columns (excluding image_name) ---
cols_old = set(df_old.columns) - {"image_name"}
cols_new = set(df_new.columns) - {"image_name"}
duplicate_cols = cols_old & cols_new

print(f"\nDuplicate columns found : {len(duplicate_cols)}")
if duplicate_cols:
    print("  Examples:")
    for c in sorted(duplicate_cols)[:10]:
        print(f"    {c}")

# --- merge on image_name, drop duplicate columns from new file ---
df_new_deduped = df_new.drop(columns=list(duplicate_cols))
df_merged = df_old.merge(df_new_deduped, on="image_name", how="inner")

print(f"\nMerged file : {df_merged.shape[0]:,} rows x {df_merged.shape[1]:,} columns")

# --- final check: no duplicate column names ---
dupes_after = df_merged.columns[df_merged.columns.duplicated()].tolist()
print(f"Duplicate columns after merge : {len(dupes_after)}")
if dupes_after:
    print(f"  {dupes_after}")

# --- save ---
df_merged.to_csv(OUT_PATH, index=False)
print(f"\nSaved to : {OUT_PATH}")

## Exploring the merge

In [ ]:
import re
import pandas as pd
import numpy as np

FEATURES_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv"

print("Loading features...")
df = pd.read_csv(FEATURES_PATH)

# clean unnamed columns
unnamed = [c for c in df.columns if str(c).startswith("Unnamed") or not str(c).strip()]
df = df.drop(columns=unnamed) if unnamed else df

all_cols = [c for c in df.columns if c != "image_name"]
df[all_cols] = df[all_cols].fillna(0)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]:,} columns\n")

# ── classify columns ──────────────────────────────────────────────────────────
grid_re = re.compile(r"_\d_\d$")

seg_global = [c for c in all_cols if c.startswith("seg_") and not grid_re.search(c)]
seg_grid   = [c for c in all_cols if c.startswith("seg_") and grid_re.search(c)]
obj_prob   = [c for c in all_cols if c.startswith("obj_") and grid_re.search(c)
              and "_count_" not in c and "_area_" not in c]
obj_count  = [c for c in all_cols if c.startswith("obj_") and "_count_" in c]
obj_area   = [c for c in all_cols if c.startswith("obj_") and "_area_" in c]
stat       = [c for c in all_cols if not c.startswith("seg_") and not c.startswith("obj_")]

groups = {
    "Segmentation global (seg_Class)":            seg_global,
    "Segmentation grid   (seg_Class_r_c)":        seg_grid,
    "Object probability  (obj_Class_r_c)":        obj_prob,
    "Object count        (obj_Class_count_r_c)":  obj_count,
    "Object area         (obj_Class_area_r_c)":   obj_area,
    "Image statistics":                            stat,
}

# ── 1. overview ───────────────────────────────────────────────────────────────
print("=" * 65)
print("1. OVERVIEW")
print("=" * 65)
print(f"\n  Total images   : {df['image_name'].nunique():,}")
print(f"  Total features : {len(all_cols):,}")
print(f"  NaN count      : {df[all_cols].isna().sum().sum():,}\n")
print(f"  {'Group':<48} {'Count':>6}  {'%':>6}")
print("  " + "-" * 63)
for name, cols in groups.items():
    pct = len(cols) / len(all_cols) * 100 if all_cols else 0
    print(f"  {name:<48} {len(cols):>6}  {pct:>5.1f}%")
print("  " + "-" * 63)
print(f"  {'TOTAL':<48} {len(all_cols):>6}")

# ── 2. image statistics ───────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("2. IMAGE STATISTICS")
print("=" * 65)
if not stat:
    print("\n  No image statistics features found.")
else:
    print(f"\n  {len(stat)} features\n")
    print(f"  {'Feature':<35} {'Mean':>10}  {'Std':>10}  {'% non-zero':>10}")
    print("  " + "-" * 70)
    for col in sorted(stat):
        nz   = (df[col] > 0).mean() * 100
        mean = df[col].mean()
        std  = df[col].std()
        print(f"  {col:<35} {mean:>10.4f}  {std:>10.4f}  {nz:>9.1f}%")

# ── 3. segmentation summary ───────────────────────────────────────────────────
print("\n" + "=" * 65)
print("3. SEGMENTATION CLASSES")
print("=" * 65)
seg_classes = sorted([c.replace("seg_", "") for c in seg_global])
print(f"\n  {len(seg_classes)} global classes  |  {len(seg_grid)} grid features\n")
print(f"  {'Class':<35} {'% non-zero':>10}  {'Mean':>10}  {'Grid cols':>10}")
print("  " + "-" * 70)
for cls in seg_classes:
    col       = f"seg_{cls}"
    nz        = (df[col] > 0).mean() * 100
    mean_v    = df[col].mean()
    grid_n    = len([c for c in seg_grid if c.startswith(f"seg_{cls}_")])
    bar       = "█" * int(nz / 5)
    print(f"  {cls:<35} {nz:>9.1f}%  {mean_v:>10.4f}  {grid_n:>10}  {bar}")

# ── 4. object detection summary ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("4. OBJECT DETECTION CLASSES")
print("=" * 65)
obj_classes = sorted(set(
    re.sub(r"_\d_\d$", "", c).replace("obj_", "", 1)
    for c in obj_prob
))
positions = sorted(set(
    re.search(r"(\d_\d)$", c).group(1)
    for c in obj_prob if re.search(r"(\d_\d)$", c)
))
print(f"\n  {len(obj_classes)} classes  |  {len(positions)} grid cells  |  variants: probability, count, area\n")
print(f"  {'Class':<30} {'% any cell':>10}  {'Mean prob':>10}")
print("  " + "-" * 55)
for cls in obj_classes:
    cls_cols = [c for c in obj_prob if re.sub(r"_\d_\d$", "", c) == f"obj_{cls}"]
    nz       = (df[cls_cols] > 0).any(axis=1).mean() * 100 if cls_cols else 0
    mean_p   = df[cls_cols].values.mean() if cls_cols else 0
    if nz > 0:
        bar = "█" * int(nz / 5)
        print(f"  {cls:<30} {nz:>9.1f}%  {mean_p:>10.4f}  {bar}")
    else:
        print(f"  {cls:<30} {nz:>9.1f}%  {mean_p:>10.4f}")

# ── 5. grid cell sparsity ─────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("5. GRID CELL SPARSITY")
print("=" * 65)
rows_ids   = sorted(set(p.split("_")[0] for p in positions))
cols_ids   = sorted(set(p.split("_")[1] for p in positions))
row_labels = {"1": "Top", "2": "Middle", "3": "Bottom"}
col_labels = {"1": "Left", "2": "Centre", "3": "Right"}

for grid_cols, title in [(seg_grid, "Segmentation"), (obj_prob, "Object probability")]:
    print(f"\n  -- {title} (mean % non-zero across classes) --\n")
    print(f"  {'':10} " + "  ".join(f"{col_labels.get(c, c):>10}" for c in cols_ids))
    print("  " + "-" * 43)
    for r in rows_ids:
        vals = []
        for c in cols_ids:
            pos_cols = [col for col in grid_cols if col.endswith(f"_{r}_{c}")]
            v = (df[pos_cols] > 0).mean(axis=0).mean() * 100 if pos_cols else 0
            vals.append(v)
        print(f"  {row_labels.get(r, r):<10} " + "  ".join(f"{v:>9.1f}%" for v in vals))

# ── 6. sparsity by group ──────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("6. SPARSITY SUMMARY BY GROUP")
print("=" * 65)
for name, cols in groups.items():
    if not cols:
        print(f"\n  {name}: no features")
        continue
    nz = (df[cols] > 0).mean(axis=0) * 100
    print(f"\n  {name}")
    print(f"    Features        : {len(cols)}")
    print(f"    Mean % non-zero : {nz.mean():.1f}%")
    print(f"    Min  % non-zero : {nz.min():.1f}%   ({nz.idxmin()})")
    print(f"    Max  % non-zero : {nz.max():.1f}%   ({nz.idxmax()})")

# ── 7. features to drop ───────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("7. FEATURES TO DROP BEFORE MODELLING")
print("=" * 65)
zv      = [c for c in all_cols if df[c].nunique() <= 1]
nzv     = [c for c in all_cols if (df[c] > 0).mean() < 0.001 and c not in zv]
to_drop = set(zv + nzv)
print(f"\n  Zero variance (nunique <= 1)  : {len(zv):,}")
print(f"  Near-zero (< 0.1% non-zero)  : {len(nzv):,}")
print(f"  Total recommended to drop    : {len(to_drop):,}")
print(f"  Remaining after drop         : {len(all_cols) - len(to_drop):,}")

# ── 8. cycling-relevant features ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("8. CYCLING-RELEVANT FEATURES CHECK")
print("=" * 65)
keywords = [
    "Bike Lane", "Bicycle", "Bicyclist", "Sidewalk",
    "Crosswalk - Plain", "Person", "Car", "Road",
    "Parking", "Street Light", "Traffic Sign (Front)",
    "Curb", "Fence", "Vegetation", "Lane Marking - General",
]
print(f"\n  {'Feature':<40} {'Global % nz':>12}  {'Grid cols':>10}  {'In drop list':>12}")
print("  " + "-" * 78)
for kw in keywords:
    global_col = f"seg_{kw}"
    grid_n     = len([c for c in seg_grid if c.startswith(f"seg_{kw}_")])
    if global_col in df.columns:
        nz        = (df[global_col] > 0).mean() * 100
        drop_flag = "YES" if global_col in to_drop else "no"
        print(f"  {global_col:<40} {nz:>11.1f}%  {grid_n:>10}  {drop_flag:>12}")
    else:
        print(f"  seg_{kw:<36} {'MISSING':>12}  {'':>10}  {'':>12}")

print("\nDiagnostic complete.")

## 1. Correlations:  All features vs human and GPT "mu"

## 4. Diagnostic of VOTES datasets: llava, gpt and human

# 4.5 GPT Votes distribution

In [ ]:
"""
GPT-5.2 Vote Breakdown — Table 2 Excel Generator
===========================================================
Computes descriptive statistics and raw distribution percentages for GPT-5.2 
votes across specified checkpoints (20k, 40k, 60k, 80k, 100k) without TrueSkill modeling.

Outputs a native Microsoft Excel file (.xlsx) directly to the specified path.
"""

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# PATHS
# ============================================================
GPT_PATH   = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_25k_20_06_26.csv"
OUTPUT_DIR = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data_09_05_26"

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
# Updated evaluation scale per user instructions
CHECKPOINTS = [20_000, 40_000, 60_000, 80_000, 100_000]

# ============================================================
# HELPERS
# ============================================================
def normalise_image_name(path: str) -> str:
    return os.path.basename(str(path).strip())


def load_votes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.dropna(subset=["vote", "image_name1", "image_name2"]).copy()
    df["image_name1"] = df["image_name1"].apply(normalise_image_name)
    df["image_name2"] = df["image_name2"].apply(normalise_image_name)
    df["vote"]        = df["vote"].astype(int)
    
    if "voting_time" in df.columns:
        df["voting_time"] = pd.to_datetime(df["voting_time"], format="mixed", dayfirst=True)
        df = df.sort_values("voting_time").reset_index(drop=True)
    return df


# ============================================================
# DATA EXTRACTION AND PROCESSING
# ============================================================
print("Loading GPT-5.2 voting dataset...")
df_gpt = load_votes(GPT_PATH)

table_rows = []

print("\nProcessing checkpoints to compile Descriptive Vote Table...")
for cp in CHECKPOINTS:
    # Slice the dataframe up to the specified checkpoint threshold safely
    slice_n = min(cp, len(df_gpt))
    df_cp = df_gpt.iloc[:slice_n].copy()
    
    # Calculate unique images involved up to this point
    unique_images = pd.concat([df_cp["image_name1"], df_cp["image_name2"]]).nunique()
    
    # Count raw decisions (1 = Left, 0 = Draw, 2 = Right)
    left_count  = (df_cp["vote"] == 1).sum()
    draw_count  = (df_cp["vote"] == 0).sum()
    right_count = (df_cp["vote"] == 2).sum()
    
    # Calculate percentages
    left_pct  = (left_count / slice_n) * 100
    draw_pct  = (draw_count / slice_n) * 100
    right_pct = (right_count / slice_n) * 100
    
    # Append structured rows descriptive of the raw data split
    table_rows.append({
        "Checkpoint Volume": f"{slice_n:,} votes",
        "Unique Images": f"{unique_images:,}",
        "Left Wins (Count)": f"{left_count:,}",
        "Left Wins (%)": f"{left_pct:.2f}%",
        "Draws (Count)": f"{draw_count:,}",
        "Draws (%)": f"{draw_pct:.2f}%",
        "Right Wins (Count)": f"{right_count:,}",
        "Right Wins (%)": f"{right_pct:.2f}%"
    })

# Convert lists into a clean pandas DataFrame
df_descriptive_table = pd.DataFrame(table_rows)

# ============================================================
# EXPORT TO NATIVE EXCEL FILE
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
excel_save_path = os.path.join(OUTPUT_DIR, "Table_2_gpt_descriptive_breakdown.xlsx")

print(f"\nWriting updated 5-step scale results to Excel sheet...")
# Save using openpyxl engine to preserve formatted layout structure
df_descriptive_table.to_excel(excel_save_path, index=False, sheet_name="Table 2 Breakdown")

print("\n" + "="*95)
print(f"Execution Complete. Excel workbook spreadsheet successfully saved to:\n {excel_save_path}")
print("="*95)

## 5. SIGMA (variation): Across data sets LLAVA, GPT AND HUMAN

filter application before comparing voting distribution

In [ ]:
# ============================================================
# HUMAN VOTE FILTERING — run this before generating figures
# ============================================================
import numpy as np
import pandas as pd
import os

HUMAN_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data_09_05_26\human_votes.csv"

MIN_VOTES_SPEED    = 5
SPEED_THRESHOLD    = 2.0
MIN_VOTES_PATTERN  = 10
MAX_STREAK         = 10
ALT_RATE_THRESHOLD = 0.90


def normalise_image_name(path):
    return os.path.basename(str(path).strip())


def compute_user_stats(df):
    counts    = df.groupby("user_id").size().rename("vote_count")
    vote_props = (
        df.groupby("user_id")["vote"]
          .value_counts(normalize=True).rename("prop").reset_index()
    )
    max_prop  = vote_props.groupby("user_id")["prop"].max().rename("max_option_prop")
    df_s      = df.sort_values(["user_id", "voting_time"]).copy()
    df_s["dt"] = df_s.groupby("user_id")["voting_time"].diff().dt.total_seconds()
    mean_dt   = df_s.groupby("user_id")["dt"].mean().fillna(np.inf).rename("mean_dt_seconds")
    return pd.concat([counts, max_prop, mean_dt], axis=1)


def compute_pattern_stats(df):
    df_s    = df.sort_values(["user_id", "voting_time"]).copy()
    results = []
    for uid, grp in df_s.groupby("user_id"):
        votes = grp["vote"].tolist()
        n     = len(votes)
        ms = cur = 1
        for i in range(1, n):
            cur = cur + 1 if votes[i] == votes[i-1] else 1
            ms  = max(ms, cur)
        tr = sum(votes[i] != votes[i-1] for i in range(1, n)) / (n-1) if n >= 2 else np.nan
        results.append({"user_id": uid, "max_streak": ms, "transition_rate": tr})
    return pd.DataFrame(results).set_index("user_id")


def compute_bias_threshold(n):
    if   n > 100: return 0.70
    elif n > 50:  return 0.75
    elif n > 10:  return 0.80
    else:         return np.nan


# load
df_raw = pd.read_csv(HUMAN_PATH)
df_raw = df_raw.dropna(subset=["vote", "image_name1", "image_name2", "voting_time"]).copy()
df_raw["image_name1"] = df_raw["image_name1"].apply(normalise_image_name)
df_raw["image_name2"] = df_raw["image_name2"].apply(normalise_image_name)
df_raw["vote"]        = df_raw["vote"].astype(int)
df_raw["voting_time"] = pd.to_datetime(df_raw["voting_time"], format="mixed", dayfirst=True)

# F1 — directional bias
s    = compute_user_stats(df_raw)
s["thr"] = s["vote_count"].apply(compute_bias_threshold)
bad  = s.index[~s["thr"].isna() & (s["max_option_prop"] >= s["thr"])].tolist()
df_f1 = df_raw[~df_raw["user_id"].isin(bad)].copy()

# F2 — speed
s   = compute_user_stats(df_f1)
bad = s.index[(s["vote_count"] >= MIN_VOTES_SPEED) &
              (s["mean_dt_seconds"] < SPEED_THRESHOLD)].tolist()
df_f2 = df_f1[~df_f1["user_id"].isin(bad)].copy()

# F3 — consecutive run
s   = compute_user_stats(df_f2).join(compute_pattern_stats(df_f2))
bad = s.index[(s["vote_count"] >= MIN_VOTES_PATTERN) &
              (s["max_streak"] >= MAX_STREAK)].tolist()
df_f3 = df_f2[~df_f2["user_id"].isin(bad)].copy()

# F4 — alternating pattern
s   = compute_user_stats(df_f3).join(compute_pattern_stats(df_f3))
bad = s.index[(s["vote_count"] >= MIN_VOTES_PATTERN) &
              (s["transition_rate"] >= ALT_RATE_THRESHOLD)].tolist()
df_human_filtered = df_f3[~df_f3["user_id"].isin(bad)].copy()

print(f"Human vote filtering complete.")
print(f"  Before : {len(df_raw):,} votes | {df_raw['user_id'].nunique():,} users")
print(f"  After  : {len(df_human_filtered):,} votes | "
      f"{df_human_filtered['user_id'].nunique():,} users")
print(f"  Removed: {len(df_raw)-len(df_human_filtered):,} votes "
      f"({(len(df_raw)-len(df_human_filtered))/len(df_raw)*100:.1f}%)")
print(f"  Filtered draw rate : {(df_human_filtered['vote']==0).mean()*100:.1f}%")

## more vote means better distribution (and better sigma)

In [ ]:
"""
GPT-5.2 as a Synthetic Voter — Publication-Quality Figures
===========================================================
Produces three separate PDF figures:
  Fig_1_vote_distribution.pdf   : Human vs GPT vote distribution (Left/Draw/Right)
  Fig_2_gpt_consistency.pdf     : GPT vote distribution across 20k / 50k / 100k votes
  Fig_3_votes_per_image.pdf     : Votes per image and sigma reduction across checkpoints

Style: Paul Tol muted palette, Helvetica/DejaVu Sans, 300 DPI, 7-inch double-column width.
All figures saved to OUTPUT_DIR.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import trueskill

warnings.filterwarnings("ignore")

# ============================================================
# PATHS — update before running
# ============================================================
GPT_PATH   = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data_09_05_26\gpt5_votes.csv"
HUMAN_PATH = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data_09_05_26\human_votes.csv"
OUTPUT_DIR = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data_09_05_26"

# ============================================================
# CHECKPOINTS
# ============================================================
CHECKPOINTS = [20_000, 50_000, 100_000]
DRAW_PROB_GPT = 0.10

# ============================================================
# PUBLICATION STYLE — Blue/Red professional palette (ColorBrewer RdBu)
# ============================================================
HUMAN_BLUE  = "#2166AC"   # deep steel blue  — Human source
GPT_RED     = "#D6604D"   # muted red        — GPT source
LEFT_BLUE   = "#4393C3"   # medium blue      — Left vote (Image A safer)
DRAW_GREY   = "#BABABA"   # neutral grey     — Draw vote (No difference)
RIGHT_RED   = "#D6604D"   # muted red        — Right vote (Image B safer)
GREY        = "#BBBBBB"   # grid lines / neutral

# Aliases kept for backward compatibility within this script
TEAL        = HUMAN_BLUE
MUTED_RED   = GPT_RED
INDIGO      = LEFT_BLUE
SAND        = DRAW_GREY
ROSE        = RIGHT_RED

CP_COLOURS = {
    20_000 : "#C6DBEF",   # light blue
    50_000 : "#4393C3",   # medium blue
    100_000: "#08306B",   # dark navy
}

FIG_WIDTH_DOUBLE = 7.0    # inches — double column
FIG_WIDTH_SINGLE = 3.5    # inches — single column
DPI              = 300
FONT_FAMILY      = "DejaVu Sans"  # closest to Helvetica available in matplotlib

# Global matplotlib style
mpl.rcParams.update({
    "font.family"        : FONT_FAMILY,
    "font.size"          : 8,
    "axes.titlesize"     : 9,
    "axes.titleweight"   : "bold",
    "axes.labelsize"     : 8,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "axes.grid"          : True,
    "axes.grid.axis"     : "y",
    "grid.color"         : GREY,
    "grid.linewidth"     : 0.5,
    "grid.alpha"         : 0.4,
    "xtick.labelsize"    : 7.5,
    "ytick.labelsize"    : 7.5,
    "legend.fontsize"    : 7.5,
    "legend.frameon"     : True,
    "legend.framealpha"  : 0.9,
    "legend.edgecolor"   : GREY,
    "figure.dpi"         : DPI,
    "savefig.dpi"        : DPI,
    "savefig.bbox"       : "tight",
    "savefig.pad_inches" : 0.05,
    "pdf.fonttype"       : 42,   # embed fonts in PDF
    "ps.fonttype"        : 42,
})

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Style and paths configured.")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  DPI        : {DPI}")
print(f"  Font       : {FONT_FAMILY}")


# ============================================================
# HELPERS
# ============================================================
def normalise_image_name(path: str) -> str:
    return os.path.basename(str(path).strip())


def load_votes(path: str, has_time: bool = True) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.dropna(subset=["vote", "image_name1", "image_name2"]).copy()
    df["image_name1"] = df["image_name1"].apply(normalise_image_name)
    df["image_name2"] = df["image_name2"].apply(normalise_image_name)
    df["vote"]        = df["vote"].astype(int)
    if has_time and "voting_time" in df.columns:
        df["voting_time"] = pd.to_datetime(df["voting_time"], format="mixed", dayfirst=True)
        df = df.sort_values("voting_time").reset_index(drop=True)
    return df


def compute_vote_dist(df: pd.DataFrame) -> dict:
    n = len(df)
    return {
        "Left"  : (df["vote"] == 1).sum() / n * 100,
        "Draw"  : (df["vote"] == 0).sum() / n * 100,
        "Right" : (df["vote"] == 2).sum() / n * 100,
    }


def compute_trueskill_ranking(df: pd.DataFrame, draw_prob: float) -> pd.DataFrame:
    env     = trueskill.TrueSkill(draw_probability=draw_prob)
    ratings = {}

    def get_r(img):
        if img not in ratings:
            ratings[img] = env.Rating()
        return ratings[img]

    for _, row in df.iterrows():
        img1, img2, vote = row["image_name1"], row["image_name2"], int(row["vote"])
        r1, r2 = get_r(img1), get_r(img2)
        if   vote == 1: ratings[img1], ratings[img2] = env.rate_1vs1(r1, r2)
        elif vote == 2: ratings[img2], ratings[img1] = env.rate_1vs1(r2, r1)
        elif vote == 0: ratings[img1], ratings[img2] = env.rate_1vs1(r1, r2, drawn=True)

    rows = [{"image_name": img, "mu": r.mu, "sigma": r.sigma,
             "ts_score": r.mu - 3 * r.sigma}
            for img, r in ratings.items()]
    return pd.DataFrame(rows)


def add_panel_label(ax, label, x=-0.12, y=1.05):
    """Add bold panel label e.g. (a) in top-left corner."""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top", ha="left")


# ============================================================
# LOAD DATA
# ============================================================
print("\nLoading data...")
df_human = load_votes(HUMAN_PATH, has_time=True)
df_gpt   = load_votes(GPT_PATH,   has_time=True)

# human: apply speed filter only (F2) to remove the most extreme noise
# for the purpose of distribution comparison we keep it simple
human_draw_rate = (df_human["vote"] == 0).mean()
gpt_draw_rate   = (df_gpt["vote"]   == 0).mean()

print(f"  Human votes : {len(df_human):,}  | draw rate: {human_draw_rate*100:.1f}%")
print(f"  GPT votes   : {len(df_gpt):,}  | draw rate: {gpt_draw_rate*100:.1f}%")

# checkpoint slices
checkpoint_dfs = {}
for cp in CHECKPOINTS:
    n = min(cp, len(df_gpt))
    checkpoint_dfs[cp] = df_gpt.iloc[:n].copy()
    print(f"  GPT checkpoint {cp:>7,} : {len(checkpoint_dfs[cp]):,} votes")


# ============================================================
# FIG 1 — Human vs GPT vote distribution
# ============================================================
print("\nGenerating Fig 1: Human vs GPT vote distribution...")

vote_options  = ["Left", "Draw", "Right"]
vote_colours  = [LEFT_BLUE, DRAW_GREY, RIGHT_RED]
human_dist    = compute_vote_dist(df_human)
gpt_dist_full = compute_vote_dist(df_gpt)

human_vals = [human_dist[v]    for v in vote_options]
gpt_vals   = [gpt_dist_full[v] for v in vote_options]

fig, ax = plt.subplots(figsize=(FIG_WIDTH_DOUBLE, 3.2))

x     = np.arange(len(vote_options))
width = 0.32

bars_h = ax.bar(x - width/2, human_vals, width, label="Human",
                color=HUMAN_BLUE, edgecolor="white", linewidth=0.6, zorder=3)
bars_g = ax.bar(x + width/2, gpt_vals,   width, label="GPT-5.2",
                color=GPT_RED, edgecolor="white", linewidth=0.6, zorder=3)

# value labels
for bar, val in zip(list(bars_h) + list(bars_g), human_vals + gpt_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{val:.1f}%",
        ha="center", va="bottom", fontsize=7, color="#333333"
    )

ax.set_xticks(x)
ax.set_xticklabels(["Left\n(Image A safer)", "Draw\n(No difference)", "Right\n(Image B safer)"])
ax.set_ylabel("Percentage of votes (%)")
ax.set_ylim(0, max(human_vals + gpt_vals) * 1.55)
ax.set_title("Vote distribution: human participants vs GPT-5.2")
ax.legend(loc="upper left")
add_panel_label(ax, "(a)")

# draw rate note — below x-axis, single line, italic, never overlaps bars
ax.text(
    0.50, -0.22,
    f"Human draw rate: {human_dist['Draw']:.1f}%   |   GPT-5.2 draw rate: {gpt_dist_full['Draw']:.1f}%",
    transform=ax.transAxes,
    fontsize=7, color="#555555",
    va="top", ha="center", style="italic"
)

fig.tight_layout()
out_path = os.path.join(OUTPUT_DIR, "Fig_1_vote_distribution.pdf")
fig.savefig(out_path)
plt.show()
print(f"  Saved: {out_path}")


# ============================================================
# FIG 2 — GPT vote distribution consistency across checkpoints
# ============================================================
print("\nGenerating Fig 2: GPT vote distribution consistency...")

checkpoint_dists = {cp: compute_vote_dist(checkpoint_dfs[cp]) for cp in CHECKPOINTS}

fig, ax = plt.subplots(figsize=(FIG_WIDTH_DOUBLE, 3.2))

x     = np.arange(len(CHECKPOINTS))
width = 0.25
offsets = [-width, 0, width]

for i, (opt, col) in enumerate(zip(vote_options, vote_colours)):
    vals = [checkpoint_dists[cp][opt] for cp in CHECKPOINTS]
    bars = ax.bar(x + offsets[i], vals, width, label=opt,
                  color=col, edgecolor="white", linewidth=0.6, zorder=3)
    for bar, val in zip(bars, vals):
        # always show label — place above bar or at fixed height for near-zero bars
        label_y = bar.get_height() + 0.5 if bar.get_height() > 1.0 else 1.5
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            label_y,
            f"{val:.1f}%",
            ha="center", va="bottom", fontsize=6.0,
            color="#333333"
        )

ax.set_xticks(x)
ax.set_xticklabels([f"{cp:,} votes" for cp in CHECKPOINTS])
ax.set_ylabel("Percentage of votes (%)")
ax.set_ylim(0, max(
    checkpoint_dists[cp][opt]
    for cp in CHECKPOINTS for opt in vote_options
) * 1.30)
ax.set_title("GPT-5.2 vote distribution stability across vote volumes")
ax.legend(loc="upper right", ncol=3)
add_panel_label(ax, "(b)")

# stability note below figure — caption style, no box, never overlaps bars
ax.text(
    0.50, -0.18,
    "Stable proportions across vote volumes indicate consistent GPT-5.2 behaviour",
    transform=ax.transAxes,
    fontsize=6.5, color="#555555",
    va="top", ha="center", style="italic"
)

fig.tight_layout()
out_path = os.path.join(OUTPUT_DIR, "Fig_2_gpt_consistency.pdf")
fig.savefig(out_path)
plt.show()
print(f"  Saved: {out_path}")


# ============================================================
# FIG 3 — Votes per image and sigma reduction
# ============================================================
print("\nGenerating Fig 3: Votes per image and sigma reduction...")
print("  Running TrueSkill at each checkpoint (this may take a few minutes)...")

ranking_results = {}
votes_per_image_stats = {}

for cp in CHECKPOINTS:
    df_cp   = checkpoint_dfs[cp]
    rank_df = compute_trueskill_ranking(df_cp, DRAW_PROB_GPT)
    ranking_results[cp] = rank_df

    vpi = pd.concat([
        df_cp[["image_name1"]].rename(columns={"image_name1": "img"}),
        df_cp[["image_name2"]].rename(columns={"image_name2": "img"}),
    ]).groupby("img").size()
    votes_per_image_stats[cp] = vpi

    print(f"  Checkpoint {cp:,}: {len(rank_df):,} images | "
          f"mean VPI={vpi.mean():.1f} | mean sigma={rank_df['sigma'].mean():.4f}")

# two-panel figure: left = votes per image, right = sigma
fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH_DOUBLE, 3.4))

# ── left: mean votes per image bar chart ──
ax = axes[0]
means_vpi = [votes_per_image_stats[cp].mean() for cp in CHECKPOINTS]
bars = ax.bar(
    [f"{cp:,}" for cp in CHECKPOINTS],
    means_vpi,
    color=[CP_COLOURS[cp] for cp in CHECKPOINTS],
    edgecolor="white", linewidth=0.6, zorder=3, width=0.5
)
for bar, val in zip(bars, means_vpi):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{val:.1f}",
        ha="center", va="bottom", fontsize=7.5, fontweight="bold"
    )
ax.set_xlabel("Vote volume")
ax.set_ylabel("Mean comparisons per image")
ax.set_title("Coverage: average votes per image")
add_panel_label(ax, "(c)")

# ── right: mean sigma bar chart ──
ax2 = axes[1]
means_sigma = [ranking_results[cp]["sigma"].mean() for cp in CHECKPOINTS]
bars2 = ax2.bar(
    [f"{cp:,}" for cp in CHECKPOINTS],
    means_sigma,
    color=[CP_COLOURS[cp] for cp in CHECKPOINTS],
    edgecolor="white", linewidth=0.6, zorder=3, width=0.5
)
for bar, val in zip(bars2, means_sigma):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{val:.3f}",
        ha="center", va="bottom", fontsize=7.5, fontweight="bold"
    )

# reduction annotation
reduction = (means_sigma[0] - means_sigma[-1]) / means_sigma[0] * 100
ax2.annotate(
    f"−{reduction:.1f}% uncertainty",
    xy=(2, means_sigma[-1]),
    xytext=(1.5, means_sigma[0] * 0.85),
    fontsize=7, color=MUTED_RED,
    arrowprops=dict(arrowstyle="->", color=MUTED_RED, lw=0.8),
)
ax2.set_xlabel("Vote volume")
ax2.set_ylabel("Mean TrueSkill sigma (σ)")
ax2.set_title("Uncertainty reduction with more votes")
add_panel_label(ax2, "(d)")

fig.tight_layout()
out_path = os.path.join(OUTPUT_DIR, "Fig_3_votes_per_image.pdf")
fig.savefig(out_path)
plt.show()
print(f"  Saved: {out_path}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("ALL FIGURES SAVED")
print("=" * 60)
print(f"  Fig_1_vote_distribution.pdf  — Human vs GPT vote distribution")
print(f"  Fig_2_gpt_consistency.pdf    — GPT consistency across checkpoints")
print(f"  Fig_3_votes_per_image.pdf    — Coverage and sigma reduction")
print(f"\n  Output folder: {OUTPUT_DIR}")
print(f"\n  Style: Paul Tol muted palette | DejaVu Sans | 300 DPI | PDF")
print(f"\n  Key findings:")
print(f"    Human draw rate : {human_draw_rate*100:.1f}%")
print(f"    GPT draw rate   : {gpt_draw_rate*100:.1f}%")
print(f"    Sigma reduction ({CHECKPOINTS[0]:,} to {CHECKPOINTS[-1]:,} votes): "
      f"{(means_sigma[0]-means_sigma[-1])/means_sigma[0]*100:.1f}%")
print(f"    Mean VPI at {CHECKPOINTS[-1]:,} votes: {votes_per_image_stats[CHECKPOINTS[-1]].mean():.1f}")

# Create diagrams of models used

In [ ]:
"""
model_diagrams_v3.py
====================
Publication-quality architecture diagrams — fully corrected, no clipping,
no overlap, compact layout.

Run:  python model_diagrams_v3.py
Output: PDF + PNG for each diagram in OUTPUT_DIR.
"""

import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

# ── paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = r"C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data_09_05_26"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DPI = 300
plt.rcParams.update({
    "font.family"     : "serif",
    "font.serif"      : ["Georgia", "Times New Roman", "DejaVu Serif"],
    "font.size"       : 8.5,
    "axes.titlesize"  : 10,
    "axes.titleweight": "bold",
    "text.usetex"     : False,
})

# palette
C_BG    = "#FFFFFF"
C_BD    = "#CBD5E1"   # border grey
C_TEXT  = "#0F172A"
C_MUTED = "#64748B"
C_BLUE  = "#1D4ED8"
C_TEAL  = "#0D9488"
C_AMBER = "#B45309"
C_GREEN = "#15803D"
C_RED   = "#B91C1C"
C_LBKG  = "#F8FAFC"


def save(fig, name):
    for ext in ["pdf", "png"]:
        p = os.path.join(OUTPUT_DIR, f"{name}.{ext}")
        fig.savefig(p, bbox_inches="tight", dpi=DPI, facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  saved: {name}")


def rbox(ax, cx, cy, w, h, fc, ec, lw=1.4, r=0.07, z=3):
    patch = FancyBboxPatch(
        (cx - w/2, cy - h/2), w, h,
        boxstyle=f"round,pad={r}",
        facecolor=fc, edgecolor=ec, linewidth=lw, zorder=z,
        clip_on=False
    )
    ax.add_patch(patch)


def txt(ax, x, y, s, size=8, color=C_TEXT, weight="normal",
        ha="center", va="center", style="normal", z=5):
    ax.text(x, y, s, ha=ha, va=va, fontsize=size,
            color=color, fontweight=weight, fontstyle=style,
            zorder=z, clip_on=False)


def arrow(ax, x0, y0, x1, y1, col=C_MUTED, lw=0.9, lbl=""):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=col,
                                lw=lw, mutation_scale=8),
                zorder=4, clip_on=False)
    if lbl:
        mx, my = (x0+x1)/2 + 0.08, (y0+y1)/2
        txt(ax, mx, my, lbl, size=6.5, color=col, style="italic", ha="left")


def line(ax, x0, y0, x1, y1, col=C_BD, lw=0.85, lbl="", z=2):
    ax.plot([x0, x1], [y0, y1], color=col, lw=lw, zorder=z, clip_on=False)
    if lbl:
        mx, my = (x0+x1)/2 + 0.08, (y0+y1)/2
        txt(ax, mx, my, lbl, size=6.5, color=C_MUTED, style="italic", ha="left")


# =============================================================================
# 1.  MLP — unchanged, already clean
# =============================================================================
def diagram_mlp():
    print("Generating: Deeper MLP")
    fig, ax = plt.subplots(figsize=(15, 7))
    fig.patch.set_facecolor(C_BG)
    ax.set_facecolor(C_BG)
    ax.set_xlim(0, 15); ax.set_ylim(0, 7); ax.axis("off")

    layers = [
        (1.2,  5, C_BLUE,  "Input",    "773 features\n|Pearson r| > 0.03"),
        (3.5,  6, C_TEAL,  "Dense 1",  "256 units\nBatchNorm + ReLU\nDropout 0.30"),
        (5.9,  5, C_TEAL,  "Dense 2",  "128 units\nBatchNorm + ReLU\nDropout 0.30"),
        (8.2,  4, C_TEAL,  "Dense 3",  "64 units\nBatchNorm + ReLU\nDropout 0.30"),
        (10.5, 3, C_TEAL,  "Dense 4",  "32 units\nReLU\nDropout 0.20"),
        (12.8, 3, C_AMBER, "Output",   "3 classes\nSoftmax\n(Low / Med / High)"),
    ]
    node_r = 0.22; y_center = 3.7; spacing = 0.65; all_pos = []
    for lx, n, col, ltop, lbot in layers:
        ys = [y_center + (i - (n-1)/2)*spacing for i in range(n)]
        pos = []
        for i, y in enumerate(ys):
            if n > 4 and i == n//2:
                txt(ax, lx, y, "...", size=11, color=C_MUTED)
                pos.append((lx, y, False)); continue
            ax.add_patch(plt.Circle((lx, y), node_r, facecolor="white",
                                    edgecolor=col, lw=1.8, zorder=4))
            ax.add_patch(plt.Circle((lx, y), node_r*0.35, facecolor=col,
                                    alpha=0.4, zorder=5))
            pos.append((lx, y, True))
        all_pos.append(pos)
        top_y = max(y for y in ys) + node_r + 0.20
        bot_y = min(y for y in ys) - node_r - 0.22
        txt(ax, lx, top_y, ltop, size=8.5, color=col, weight="bold", va="bottom")
        txt(ax, lx, bot_y, lbot, size=7.0, color=C_MUTED, va="top")

    for i in range(len(all_pos)-1):
        src = [(x,y) for x,y,r in all_pos[i]   if r]
        dst = [(x,y) for x,y,r in all_pos[i+1] if r]
        for (x0,y0) in src:
            for (x1,y1) in dst:
                ax.plot([x0,x1],[y0,y1], color=C_BD, lw=0.40, alpha=0.50, zorder=1)

    txt(ax, 7.0, 0.28,
        "Optimizer: Adam(lr=0.001)  |  Loss: Sparse Categorical CE  |  "
        "Sample weights: $w_i = 1/\\sigma_i^2$  |  "
        "Early stopping (patience=20)  |  ReduceLROnPlateau",
        size=7.5, color=C_MUTED, style="italic")
    ax.set_title(
        "Deeper MLP — Sequential Architecture\n"
        "3-partition x 5-fold stratified CV  |  Sigma-weighted training",
        color=C_TEXT, pad=6, fontsize=10)
    fig.tight_layout(pad=0.8)
    save(fig, "Fig1_MLP_Architecture")


# =============================================================================
# 2.  XGBOOST — fully redrawn, compact, nothing clips
# =============================================================================
def diagram_xgboost():
    print("Generating: XGBoost")

    # canvas: x=[0,12], y=[0,8]
    # tree occupies x=[1.2, 10.5], level-wise annotation at x=11.0
    # depth labels at x=0.15
    W, H = 12, 8
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor(C_BG)
    ax.set_facecolor(C_BG)
    ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")

    bw, bh = 1.70, 0.50

    # y positions with enough room: footer at y=0.5, title at top
    d0y, d1y, d2y, d3y = 6.8, 5.2, 3.5, 1.8

    # x positions — 8 leaves fit in [1.2, 10.5] with gap 0.15 between boxes
    # leaf width 1.1, so each leaf+gap = 1.25
    # 8 leaves * 1.25 = 10.0 → fits in [1.0, 11.0]
    lw, lh = 1.05, 0.42
    leaf_gap = 0.20
    leaf_step = lw + leaf_gap
    leaf_start = 1.10
    d3_xs = [leaf_start + i * leaf_step for i in range(8)]  # 8 leaves

    # depth 2: 4 nodes, each centred over their 2 children
    d2_xs = [(d3_xs[2*i] + d3_xs[2*i+1])/2 for i in range(4)]

    # depth 1: 2 nodes, centred over their 4 children
    d1_xs = [(d2_xs[0]+d2_xs[1])/2, (d2_xs[2]+d2_xs[3])/2]

    # depth 0: root centred over all
    d0_x  = (d1_xs[0]+d1_xs[1])/2

    # draw nodes
    rbox(ax, d0_x, d0y, bw, bh, C_LBKG, C_BLUE, lw=1.8)
    txt(ax, d0_x, d0y, "Root split\n(best global feature)", size=8, weight="bold")

    for dx, lbl in zip(d1_xs, ["Split (depth 1)\nLeft branch",
                                 "Split (depth 1)\nRight branch"]):
        rbox(ax, dx, d1y, bw, bh, C_LBKG, C_BLUE, lw=1.8)
        txt(ax, dx, d1y, lbl, size=8, weight="bold")

    for dx in d2_xs:
        rbox(ax, dx, d2y, bw, bh, C_LBKG, C_BLUE, lw=1.8)
        txt(ax, dx, d2y, "Split (depth 2)", size=8, weight="bold")

    for dx in d3_xs:
        rbox(ax, dx, d3y, lw, lh, "#F0FDF4", C_GREEN, lw=1.4)
        txt(ax, dx, d3y, "$w_j$", size=8, color=C_GREEN, weight="bold")

    # edges: root -> depth1
    for d1x, lbl in zip(d1_xs, ["≤ t", "> t"]):
        line(ax, d0_x, d0y-bh/2, d1x, d1y+bh/2, lbl=lbl)

    # depth1 -> depth2
    for pi, (d1x, pairs) in enumerate(zip(d1_xs, [(0,1),(2,3)])):
        for ci, lbl in zip(pairs, ["≤ t", "> t"]):
            line(ax, d1x, d1y-bh/2, d2_xs[ci], d2y+bh/2, lbl=lbl)

    # depth2 -> leaves
    for pi, d2x in enumerate(d2_xs):
        for li in [2*pi, 2*pi+1]:
            line(ax, d2x, d2y-bh/2, d3_xs[li], d3y+lh/2)

    # depth labels — left margin, clear of nodes
    for depth, dy, lbl in [
        (0, d0y, "Depth 0"),
        (1, d1y, "Depth 1"),
        (2, d2y, "Depth 2"),
        (3, d3y, "Depth 3 (leaves)"),
    ]:
        txt(ax, 0.40, dy, lbl, size=7.5, color=C_MUTED,
            style="italic", ha="right", va="center")
        ax.plot([0.42, 0.55], [dy, dy], color=C_BD, lw=0.7,
                linestyle="--", alpha=0.6, zorder=1)

    # level-wise annotation — right margin, well inside canvas
    arrow(ax, 10.7, d0y, 10.7, d3y+0.3, col=C_RED, lw=1.4)
    txt(ax, 11.0, (d0y+d3y)/2,
        "Level-wise\ngrowth:\nall nodes at\ndepth d\nbefore d+1",
        size=7.5, color=C_RED, weight="bold", ha="left", va="center")

    # footer
    txt(ax, W/2, 0.45,
        r"Prediction: $\hat{y}_i = \sum_{k=1}^{K} f_k(x_i)$   |   "
        r"Objective: $\mathcal{L} + \Omega(f_k)$   |   "
        "max_depth=8   n_estimators=400   subsample=0.8",
        size=7.5, color=C_MUTED, style="italic")

    ax.set_title(
        "XGBoost — Level-wise Tree Growth\n"
        "Regularised gradient boosting  |  σ²-weighted samples  |  3-class output",
        color=C_TEXT, pad=6, fontsize=10)
    fig.tight_layout(pad=0.8)
    save(fig, "Fig2_XGBoost_Architecture")


# =============================================================================
# 3.  LIGHTGBM — fully redrawn
# =============================================================================
def diagram_lightgbm():
    print("Generating: LightGBM")

    W, H = 11, 8.5
    fig, ax = plt.subplots(figsize=(W, H))
    fig.patch.set_facecolor(C_BG)
    ax.set_facecolor(C_BG)
    ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")

    bw, bh = 2.00, 0.50

    # layout: tree on right side, histogram inset bottom-left
    # y levels
    y0, y1, y2, y3 = 7.4, 5.8, 4.1, 2.4

    # x positions — kept well inside canvas [1.0, 10.0]
    # root: 5.5
    # left leaf: 2.5  (stays a leaf)
    # right (expanded): 8.0
    # R_R_L (leaf): 6.5  R_R_R (expanded): 9.2
    # R_R_R_L (leaf): 8.2  R_R_R_R (leaf): 10.2 -- 10.2 clips if bw=2, so centre at 9.8
    nodes = {
        "R":       (5.5, y0),
        "R_L":     (2.5, y1),
        "R_R":     (8.0, y1),
        "R_R_L":   (6.4, y2),
        "R_R_R":   (9.2, y2),
        "R_R_R_L": (8.0, y3),
        "R_R_R_R": (9.8, y3),
    }
    leaves  = {"R_L", "R_R_L", "R_R_R_L", "R_R_R_R"}
    focused = {"R_R", "R_R_R"}

    def style(k):
        if k in focused:  return "#FEF3C7", C_RED,   2.0
        if k in leaves:   return "#F0FDF4", C_GREEN, 1.5
        return C_LBKG, C_BLUE, 1.8

    def label(k):
        if k in focused: return "Max $\\Delta$ Loss\n\u2192 expand next"
        if k in leaves:  return "Leaf  $w_j$"
        return "Split node"

    for k, (cx, cy) in nodes.items():
        fc, ec, lw = style(k)
        rbox(ax, cx, cy, bw, bh, fc, ec, lw=lw)
        txt(ax, cx, cy, label(k), size=8, weight="bold",
            color=ec if k in leaves else C_TEXT)

    # edges
    edges = [
        ("R", "R_L", y0, y1, "≤ t"),
        ("R", "R_R", y0, y1, "> t"),
        ("R_R", "R_R_L",  y1, y2, "≤ t"),
        ("R_R", "R_R_R",  y1, y2, "> t"),
        ("R_R_R", "R_R_R_L", y2, y3, "≤ t"),
        ("R_R_R", "R_R_R_R", y2, y3, "> t"),
    ]
    for src, dst, sy, dy, lbl in edges:
        cx0, cy0 = nodes[src]
        cx1, cy1 = nodes[dst]
        line(ax, cx0, cy0-bh/2, cx1, cy1+bh/2, lbl=lbl)

    # growth step labels — right of focused nodes
    for step, k in enumerate(["R", "R_R", "R_R_R"], 1):
        cx, cy = nodes[k]
        txt(ax, cx + bw/2 + 0.25, cy, f"Step {step}",
            size=8, color=C_RED, weight="bold", ha="left")

    # histogram inset — bottom-left, sized to not waste space
    rbox(ax, 2.0, 1.8, 3.5, 1.3, "#EFF6FF", C_BLUE, lw=1.0)
    txt(ax, 2.0, 1.95, "Histogram binning",
        size=8, color=C_BLUE, weight="bold")
    txt(ax, 2.0, 1.70,
        "Continuous features are discretised\n"
        "into bins before each split search.\n"
        "Reduces complexity vs exact search.",
        size=6.8, color=C_MUTED)

    # footer
    txt(ax, W/2, 0.35,
        "num_leaves=63   n_estimators=300   subsample=0.8   "
        "Always expand the leaf with the maximum loss reduction",
        size=7.5, color=C_MUTED, style="italic")

    ax.set_title(
        "LightGBM — Leaf-wise (Best-first) Tree Growth\n"
        "Histogram-based splits  |  σ²-weighted samples  |  3-class output",
        color=C_TEXT, pad=6, fontsize=10)
    fig.tight_layout(pad=0.8)
    save(fig, "Fig3_LightGBM_Architecture")


# =============================================================================
# 4.  CATBOOST — fully redrawn
# =============================================================================
def diagram_catboost():
    print("Generating: CatBoost")

    W, H = 13, 9
    fig, ax = plt.subplots(figsize=(W, H))
    fig.patch.set_facecolor(C_BG)
    ax.set_facecolor(C_BG)
    ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")

    bw, bh = 1.60, 0.48
    lw_leaf, lh_leaf = 1.10, 0.42

    # y levels — room for title, callout, tree, footer
    d0y, d1y, d2y, d3y = 7.4, 5.7, 3.9, 2.0

    # 8 leaves fit inside [0.8, 12.2]
    # leaf_step = lw_leaf + gap = 1.10 + 0.30 = 1.40
    # 8 leaves: start=0.80, end=0.80+7*1.40=10.60 → centre of last = 10.60
    leaf_step = lw_leaf + 0.30
    leaf_start = 0.80
    d3_xs = [leaf_start + i * leaf_step for i in range(8)]

    # depth 2: 4 nodes centred over pairs of leaves
    d2_xs = [(d3_xs[2*i] + d3_xs[2*i+1])/2 for i in range(4)]

    # depth 1: 2 nodes centred over pairs of depth-2 nodes
    d1_xs = [(d2_xs[0]+d2_xs[1])/2, (d2_xs[2]+d2_xs[3])/2]

    # depth 0: root
    d0_x = (d1_xs[0]+d1_xs[1])/2

    # draw nodes
    rbox(ax, d0_x, d0y, bw, bh, C_LBKG, C_BLUE, lw=1.8)
    txt(ax, d0_x, d0y, "Split", size=8.5, weight="bold")

    for dx in d1_xs:
        rbox(ax, dx, d1y, bw, bh, C_LBKG, C_BLUE, lw=1.8)
        txt(ax, dx, d1y, "Split", size=8.5, weight="bold")

    for dx in d2_xs:
        rbox(ax, dx, d2y, bw, bh, C_LBKG, C_BLUE, lw=1.8)
        txt(ax, dx, d2y, "Split", size=8.5, weight="bold")

    for dx in d3_xs:
        rbox(ax, dx, d3y, lw_leaf, lh_leaf, "#F0FDF4", C_GREEN, lw=1.4)
        txt(ax, dx, d3y, "$w_j$", size=8, color=C_GREEN, weight="bold")

    # edges root → depth 1
    for dx, lbl in zip(d1_xs, ["≤", ">"]):
        line(ax, d0_x, d0y-bh/2, dx, d1y+bh/2, lbl=lbl)

    # edges depth 1 → depth 2
    for pi, d1x in enumerate(d1_xs):
        for ci, lbl in zip([2*pi, 2*pi+1], ["≤", ">"]):
            line(ax, d1x, d1y-bh/2, d2_xs[ci], d2y+bh/2, lbl=lbl)

    # edges depth 2 → leaves
    for pi, d2x in enumerate(d2_xs):
        for li, lbl in zip([2*pi, 2*pi+1], ["≤", ">"]):
            line(ax, d2x, d2y-bh/2, d3_xs[li], d3y+lh_leaf/2, lbl=lbl)

    # same-depth dashed lines and feature labels
    split_info = {
        d0y: ("Feature $f_1 \\leq t_1$", [d0_x]),
        d1y: ("Feature $f_2 \\leq t_2$", d1_xs),
        d2y: ("Feature $f_3 \\leq t_3$", d2_xs),
    }
    for dy, (flbl, xs) in split_info.items():
        x_left  = min(xs) - bw/2 - 0.25
        x_right = max(xs) + bw/2 + 0.25
        ax.plot([x_left, x_right], [dy, dy],
                color=C_BLUE, lw=0.9, linestyle="--", alpha=0.40, zorder=1)
        # label placed to the right, but capped so it doesn't go off canvas
        lx = min(x_right + 0.15, W - 0.05)
        txt(ax, lx, dy, flbl, size=7.5, color=C_BLUE,
            weight="bold", ha="left", va="center")

    # callout box — positioned in left margin BELOW title, clear of tree
    # title is at top, callout at y=8.20 works if tree top is d0y=7.4
    rbox(ax, 2.0, 8.30, 3.6, 0.65, "#EFF6FF", C_BLUE, lw=1.0)
    txt(ax, 2.0, 8.30,
        "Key property: all nodes at the\nsame depth use the SAME split",
        size=7.5, color=C_BLUE, weight="bold")

    # footer
    txt(ax, W/2, 0.50,
        "depth=6   iterations=300   "
        "Symmetric structure: fast scoring, built-in regularisation via balanced trees",
        size=7.5, color=C_MUTED, style="italic")

    ax.set_title(
        "CatBoost — Symmetric (Oblivious) Trees\n"
        "Same split per depth level  |  σ²-weighted samples  |  3-class output",
        color=C_TEXT, pad=6, fontsize=10)
    fig.tight_layout(pad=0.8)
    save(fig, "Fig4_CatBoost_Architecture")


# =============================================================================
if __name__ == "__main__":
    print("=" * 65)
    print("Generating publication-quality model architecture diagrams")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 65)
    diagram_mlp()
    diagram_xgboost()
    diagram_lightgbm()
    diagram_catboost()
    print("=" * 65)
    print("Done. Four PDF + PNG pairs saved.")
    print("=" * 65)

# Figure for paper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

models    = ['Deeper MLP', 'XGBoost', 'LightGBM', 'CatBoost']
gpt_means = [0.6170, 0.6324, 0.6355, 0.6287]
gpt_stds  = [0.0079, 0.0072, 0.0074, 0.0062]
hum_means = [0.4183, 0.4233, 0.4218, 0.4269]
hum_stds  = [0.0084, 0.0105, 0.0061, 0.0022]
baseline  = 1 / 3

BLUE  = '#2171B5'
GREEN = '#1B7837'

x     = np.arange(len(models))
width = 0.32

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars_gpt = ax.bar(
    x - width / 2, gpt_means, width,
    yerr=gpt_stds, capsize=4,
    color=BLUE, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    label='GPT-5.2 (200k votes)'
)

bars_hum = ax.bar(
    x + width / 2, hum_means, width,
    yerr=hum_stds, capsize=4,
    color=GREEN, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    label='Human (19,729 filtered votes)'
)

for bar, mean, std in zip(bars_gpt, gpt_means, gpt_stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        mean + std + 0.007,
        f'{mean*100:.1f}%',
        ha='center', va='bottom',
        fontsize=8.5, fontweight='bold',
        color=BLUE
    )

for bar, mean, std in zip(bars_hum, hum_means, hum_stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        mean + std + 0.007,
        f'{mean*100:.1f}%',
        ha='center', va='bottom',
        fontsize=8.5, fontweight='bold',
        color=GREEN
    )

ax.axhline(
    baseline,
    color='#c0392b', linestyle='--',
    linewidth=1.6, alpha=0.9,
    label=f'Random baseline ({baseline*100:.1f}%)'
)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Balanced accuracy (%)', fontsize=11)
ax.set_ylim(0, 0.74)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v*100:.0f}%')
)
ax.set_title(
    'Safety score prediction: balanced accuracy by model and dataset\n'
    '3-class qcut | 1/\u03c3\u00b2 sample weighting | Location-level GroupKFold | 3\u00d75 CV',
    fontsize=11, fontweight='bold', pad=10
)
ax.legend(
    fontsize=10, frameon=True, framealpha=0.95,
    loc='lower right', edgecolor='#cccccc'
)
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.25, linestyle='--', color='#aaaaaa')
ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

Another chart for paper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

BLUE   = '#2171B5'
ORANGE = '#E8743B'

BIN_LABELS = ['Low', 'Medium', 'High']

# ── data from classification report ──────────────────────────────────────────

# GPT LightGBM (best GPT model, BA = 62.8%)
gpt_recall    = [0.679, 0.555, 0.650]
gpt_precision = [0.725, 0.499, 0.684]
gpt_f1        = [0.702, 0.525, 0.667]
gpt_support   = [1331,  1331,  1331]

# Human CatBoost (best Human model, BA = 42.7%)
hum_recall    = [0.365, 0.475, 0.441]
hum_precision = [0.494, 0.355, 0.477]
hum_f1        = [0.420, 0.406, 0.458]
hum_support   = [1148,  1148,  1148]


def recall_to_cm(recall, support):
    n  = len(recall)
    cm = np.zeros((n, n))
    for i in range(n):
        tp  = round(recall[i] * support[i])
        rem = support[i] - tp
        cm[i, i] = tp
        for j in range(n):
            if j != i:
                cm[i, j] = rem / (n - 1)
    return cm.astype(int)


gpt_cm = recall_to_cm(gpt_recall, gpt_support)
hum_cm = recall_to_cm(hum_recall, hum_support)

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('white')
fig.suptitle(
    'Confusion matrices — best model per dataset\n'
    'Row-normalised: diagonal = recall per class',
    fontsize=12, fontweight='bold', y=1.02
)

configs = [
    (axes[0], gpt_cm, gpt_recall, gpt_support,
     'GPT — LightGBM  (BA = 62.8%)', BLUE),
    (axes[1], hum_cm, hum_recall, hum_support,
     'Human — CatBoost  (BA = 42.7%)', ORANGE),
]

for ax, cm, recall, support, title, color in configs:
    ax.set_facecolor('white')

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_n     = np.where(row_sums > 0, cm / row_sums, 0.0)

    im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1, aspect='equal')

    for i in range(3):
        for j in range(3):
            val = cm_n[i, j]
            raw = int(cm[i, j])
            ax.text(j, i, f'{val:.2f}\n({raw})',
                    ha='center', va='center', fontsize=9,
                    color='white' if val > 0.55 else '#222222')

    for i in range(3):
        ax.add_patch(plt.Rectangle(
            (i - 0.5, i - 0.5), 1, 1,
            fill=False, edgecolor='#c0392b', linewidth=2.5
        ))

    # recall labels placed further right to avoid colorbar overlap
    for i, (r, sup) in enumerate(zip(recall, support)):
        ax.text(3.55, i,
                f'recall={r:.3f}\nn={sup:,}',
                ha='left', va='center',
                fontsize=7.5, color='#444444',
                transform=ax.transData)

    cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.15)
    cbar.set_label('Recall (row-normalised)', fontsize=8)
    cbar.ax.tick_params(labelsize=7.5)

    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(BIN_LABELS, fontsize=10)
    ax.set_yticklabels(BIN_LABELS, fontsize=10)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True', fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=10, color='#111111')
    ax.tick_params(length=0)

    for k in [0.5, 1.5]:
        ax.axhline(k, color='white', linewidth=1.5)
        ax.axvline(k, color='white', linewidth=1.5)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'confusion_matrices_best_models.png')
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

ablation chart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

BLUE  = '#2171B5'
GREEN = '#1B7837'

OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'

# ── data: updated 200k results ────────────────────────────────────────────────
families = [
    'Composite', 'Seg coverage', 'Seg centroid X', 'Seg centroid Y',
    'Seg perimeter', 'Seg grid', 'Obj probability', 'Obj count',
    'Obj area', 'Image statistics'
]
gpt_drop = [+0.0027, -0.0007, +0.0003, +0.0041,
            +0.0016, +0.0044, +0.0013, -0.0001,
            -0.0004, +0.0020]
hum_drop = [+0.0021, +0.0028, -0.0013, +0.0019,
            +0.0019, +0.0009, +0.0014, -0.0013,
            +0.0017, -0.0005]

# sort by average absolute drop descending
avg_abs    = [(abs(g) + abs(h)) / 2 for g, h in zip(gpt_drop, hum_drop)]
order      = sorted(range(len(families)), key=lambda i: avg_abs[i], reverse=True)
families_s = [families[i] for i in order]
gpt_s      = [gpt_drop[i] for i in order]
hum_s      = [hum_drop[i] for i in order]

x     = np.arange(len(families_s))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars_gpt = ax.bar(
    x - width / 2, gpt_s, width,
    color=BLUE, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    label='GPT-5.2 (LightGBM, BA = 62.8%)'
)
bars_hum = ax.bar(
    x + width / 2, hum_s, width,
    color=GREEN, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    label='Human (CatBoost, BA = 42.7%)'
)

y_max = max(max(gpt_s), max(hum_s))
y_min = min(min(gpt_s), min(hum_s))

for bar, val in zip(bars_gpt, gpt_s):
    offset = y_max * 0.04 if val >= 0 else y_min * 0.04
    va     = 'bottom' if val >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f'{val*100:+.2f} pp',
            ha='center', va=va,
            fontsize=7, color='#333333', fontweight='bold')

for bar, val in zip(bars_hum, hum_s):
    offset = y_max * 0.04 if val >= 0 else y_min * 0.04
    va     = 'bottom' if val >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f'{val*100:+.2f} pp',
            ha='center', va=va,
            fontsize=7, color='#333333', fontweight='bold')

ax.axhline(0, color='#333333', linewidth=1.0, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(families_s, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Balanced accuracy drop (pp)', fontsize=10)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v*100:+.2f} pp')
)
ax.set_title(
    'Ablation B1 — Leave-one-group-out feature family contribution\n'
    'Positive = removing this family reduces balanced accuracy  |  '
    'Negative = removing this family has no cost or marginal benefit',
    fontsize=11, fontweight='bold', pad=25
)
ax.legend(fontsize=9, frameon=True, framealpha=0.95,
          loc='upper right', edgecolor='#cccccc')
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax.tick_params(labelsize=9)
ax.set_ylim(y_min * 1.5, y_max * 1.6)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'ablation_b1_200k.pdf')
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

# Image year distribution

In [ ]:
import json
import pandas as pd
import os
from collections import Counter

import pandas as pd

COORDS_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\metadata_images_mapping_corrected.csv'

coords_df = pd.read_csv(COORDS_PATH)
coords_df['image_name'] = coords_df['image_blob_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
coords_df['point_id'] = coords_df['image_name'].str.extract(r'^(\d+)_')

from azure.storage.blob import BlobServiceClient

AZURE_CONN_STR = (
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client('london')

OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'
os.makedirs(OUTPUT_DIR, exist_ok=True)

unique_blobs = (
    coords_df[['point_id', 'metadata_blob_name']]
    .drop_duplicates(subset='point_id')
    .reset_index(drop=True)
)

results = []
errors  = 0

for i, row in unique_blobs.iterrows():
    pid           = row['point_id']
    metadata_blob = row['metadata_blob_name']
    try:
        data     = container_client.download_blob(metadata_blob).readall()
        meta     = json.loads(data)
        date_str = meta.get('metadata_response', {}).get('date', '')
        if date_str:
            results.append({'point_id': pid, 'year': int(date_str[:4])})
    except Exception:
        errors += 1
    if (i + 1) % 1000 == 0:
        print(f'  Processed {i+1:,} / {len(unique_blobs):,}')

years_df = pd.DataFrame(results)

RIVER_EXCLUDE_IDS = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '4943', '4354', '1610', '3822', '1984'
]
years_df_clean = years_df[~years_df['point_id'].isin(RIVER_EXCLUDE_IDS)].copy()
years_df_clean.to_csv(os.path.join(OUTPUT_DIR, 'image_capture_years.csv'), index=False)

print(f'Locations processed         : {len(unique_blobs):,}')
print(f'Years extracted             : {len(years_df):,}')
print(f'Errors                      : {errors}')
print(f'After excluding river points: {len(years_df_clean):,}')
print(f'Saved to                    : image_capture_years.csv')

In [ ]:
import pandas as pd
import os

FEATURES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv'

feat_df = pd.read_csv(FEATURES_PATH, usecols=['image_name'])
feat_df['image_name'] = feat_df['image_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
feat_df['point_id'] = feat_df['image_name'].str.extract(r'^(\d+)_')

print(f'Total images in features file : {len(feat_df):,}')

# merge each image with its location's year, using the FULL years_df
# (no river exclusion here, this chart describes all captured images)
images_with_year = feat_df.merge(years_df, on='point_id', how='inner')

print(f'Images matched with a year    : {len(images_with_year):,}')
print(f'Images not matched            : {len(feat_df) - len(images_with_year):,}')

In [ ]:
from collections import Counter

year_counts  = Counter(images_with_year['year'].tolist())
sorted_years = sorted(year_counts.keys())
total        = len(images_with_year)

print(f'{"Year":<8}{"Count":>10}{"Share":>10}')
print('-' * 28)
for y in sorted_years:
    n = year_counts[y]
    print(f'{y:<8}{n:>10,}{n/total*100:>9.1f}%')
print('-' * 28)
print(f'{"Total":<8}{total:>10,}{100.0:>9.1f}%')
print(f'\nYear range  : {min(sorted_years)} to {max(sorted_years)}')
print(f'Median year : {int(images_with_year["year"].median())}')
print(f'Most common : {year_counts.most_common(1)[0][0]} '
      f'({year_counts.most_common(1)[0][1]:,} images, '
      f'{year_counts.most_common(1)[0][1]/total*100:.1f}%)')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE        = '#2171B5'
FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

counts = [year_counts[y] for y in sorted_years]
pcts   = [c / total * 100 for c in counts]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars = ax.bar(
    sorted_years, counts,
    color=BLUE, alpha=0.85,
    edgecolor='white', linewidth=0.5,
    width=0.6
)

for bar, val, pct in zip(bars, counts, pcts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts) * 0.008,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=8, fontweight='bold', color='#333333',
        linespacing=1.4
    )

ax2 = ax.twinx()
ax2.set_ylim(0, ax.get_ylim()[1] / total * 100)
ax2.set_ylabel('Share of images (%)', fontsize=10, color='#555555')
ax2.tick_params(labelsize=9, colors='#555555')
ax2.spines[['top', 'right']].set_visible(True)
ax2.spines['right'].set_color('#cccccc')
ax2.spines['top'].set_visible(False)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

ax.set_xlabel('Capture year', fontsize=10)
ax.set_ylabel('Number of images', fontsize=10)
ax.set_title(
    'Google Street View image capture year distribution\n'
    f'Total: {total:,} images across {len(sorted_years)} capture years',
    fontsize=11, fontweight='bold', pad=22
)
ax.set_xticks(sorted_years)
ax.set_xticklabels([str(y) for y in sorted_years], fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax.tick_params(labelsize=9)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'image_capture_year_distribution.pdf')
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

# second map of the previous one

In [ ]:
import pandas as pd
import os

COORDS_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\metadata_images_mapping_corrected.csv'

coords_df = pd.read_csv(COORDS_PATH)
coords_df['image_name'] = coords_df['image_blob_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
coords_df['point_id'] = coords_df['image_name'].str.extract(r'^(\d+)_')

RIVER_EXCLUDE_IDS = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '4943', '4354', '1610', '3822', '1984'
]

point_coords = (
    coords_df.groupby('point_id')[['actual_lat', 'actual_lng']]
    .first().reset_index()
    .rename(columns={'actual_lat': 'lat', 'actual_lng': 'lng'})
)
point_coords = point_coords[
    ~point_coords['point_id'].isin(RIVER_EXCLUDE_IDS)
].reset_index(drop=True)

print(f'point_coords ready: {len(point_coords):,} locations')

In [ ]:
from collections import Counter

FEATURES_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\features\processed_features_merged_02_06_26.csv'

feat_df = pd.read_csv(FEATURES_PATH, usecols=['image_name'])
feat_df['image_name'] = feat_df['image_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
feat_df['point_id'] = feat_df['image_name'].str.extract(r'^(\d+)_').astype(str)

years_df_all = pd.read_csv(
    r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\image_capture_years_all.csv'
)
years_df_all = years_df_all[years_df_all['point_id'] != 'nan'].copy()
years_df_all['point_id'] = years_df_all['point_id'].astype(str)

images_with_year = feat_df.merge(years_df_all, on='point_id', how='inner')
year_counts      = Counter(images_with_year['year'].tolist())
sorted_years     = sorted(year_counts.keys())
total            = len(images_with_year)

print(f'images_with_year : {total:,}')
print(f'years            : {sorted_years}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import geopandas as gpd
import contextily as ctx
import numpy as np
import os, io, requests
from shapely.ops import unary_union

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'

year_map_df = images_with_year.merge(
    point_coords, on='point_id', how='inner'
).drop_duplicates(subset='point_id')

print(f'Locations for map: {len(year_map_df):,}')

gdf_years = gpd.GeoDataFrame(
    year_map_df,
    geometry=gpd.points_from_xy(year_map_df['lng'], year_map_df['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:3857')

london_boundary = None
try:
    url     = ('https://raw.githubusercontent.com/martinjc/UK-GeoJSON/'
               'master/json/administrative/eng/lad.json')
    resp    = requests.get(url, timeout=20)
    gdf_raw = gpd.read_file(io.BytesIO(resp.content))
    london  = gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].copy()
    london_boundary = london.dissolve().to_crs('EPSG:3857')
    boroughs        = london.to_crs('EPSG:3857')
    print('London boundary loaded.')
except Exception as e:
    print(f'Boundary failed: {e}')
    boroughs = None

year_min = int(year_map_df['year'].min())
year_max = int(year_map_df['year'].max())
norm     = mcolors.Normalize(vmin=year_min, vmax=year_max)
cmap     = cm.viridis

fig, (ax_bar, ax_map) = plt.subplots(
    1, 2, figsize=(22, 10),
    gridspec_kw={'wspace': 0.08}
)
fig.patch.set_facecolor('white')
fig.subplots_adjust(top=0.88)

# ── Panel A ───────────────────────────────────────────────────────────────────
ax_bar.set_facecolor('white')

counts_list = [year_counts[y] for y in sorted_years]
pcts_list   = [c / total * 100 for c in counts_list]
bar_colors  = [cmap(norm(y)) for y in sorted_years]
ylim_max    = max(counts_list) * 1.22

bars = ax_bar.bar(
    sorted_years, counts_list,
    color=bar_colors, alpha=0.9,
    edgecolor='white', linewidth=0.5,
    width=0.6
)

for bar, val, pct in zip(bars, counts_list, pcts_list):
    ax_bar.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts_list) * 0.008,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=7, fontweight='bold', color='#333333',
        linespacing=1.4
    )

ax2 = ax_bar.twinx()
ax2.set_ylabel('Share of images (%)', fontsize=10, color='#555555')
ax2.tick_params(labelsize=9, colors='#555555')
ax2.spines[['top']].set_visible(False)
ax2.spines['right'].set_color('#cccccc')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax2.set_ylim(0, ylim_max / total * 100)
ax2.autoscale(False)

ax_bar.set_xlabel('Capture year', fontsize=10)
ax_bar.set_ylabel('Number of images', fontsize=10)
ax_bar.set_xticks(sorted_years)
ax_bar.set_xticklabels([str(y) for y in sorted_years], fontsize=8, rotation=45, ha='right')
ax_bar.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax_bar.spines[['top', 'right']].set_visible(False)
ax_bar.spines['left'].set_color('#cccccc')
ax_bar.spines['bottom'].set_color('#cccccc')
ax_bar.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax_bar.tick_params(labelsize=9)
ax_bar.set_ylim(0, ylim_max)
ax_bar.autoscale(False)
ax_bar.set_title(
    'Panel A — Google Street View image capture year distribution\n'
    f'Total: {total:,} images across {len(sorted_years)} capture years',
    fontsize=11, fontweight='bold', loc='left', pad=10
)

# ── Panel B ───────────────────────────────────────────────────────────────────
ax_map.set_facecolor('white')
ax_map.patch.set_facecolor('white')

xlim = (-57000, 42000)
ylim = (6670000, 6746000)
ax_map.set_xlim(xlim)
ax_map.set_ylim(ylim)

# light grey London fill and borough lines — no basemap for clean white bg
if london_boundary is not None:
    london_boundary.plot(ax=ax_map, color='#f0f0f0',
                         edgecolor='none', zorder=1)
if boroughs is not None:
    boroughs.boundary.plot(ax=ax_map, color='#cccccc',
                           linewidth=0.5, zorder=2)

# scatter points
sc = ax_map.scatter(
    gdf_years.geometry.x,
    gdf_years.geometry.y,
    c=gdf_years['year'],
    cmap=cmap, norm=norm,
    s=14, alpha=0.85,
    linewidths=0, rasterized=True, zorder=5
)

# outer London boundary
if london_boundary is not None:
    london_boundary.boundary.plot(
        ax=ax_map, color='#333333',
        linewidth=1.2, alpha=1.0, zorder=6
    )

ax_map.set_xlim(xlim)
ax_map.set_ylim(ylim)
ax_map.set_axis_off()
ax_map.set_title(
    'Panel B — Spatial distribution of Street View images by capture year',
    fontsize=11, fontweight='bold', loc='left', pad=10
)

# colorbar using exact figure coordinates from diagnostic
cax  = fig.add_axes([0.910, 0.160, 0.012, 0.655])
cbar = fig.colorbar(sc, cax=cax, orientation='vertical')
cbar.set_label('Capture year', fontsize=10)
cbar.set_ticks(range(year_min, year_max + 1, 2))
cbar.ax.tick_params(labelsize=8)

out_path = os.path.join(OUTPUT_DIR, 'image_year_distribution_panels.pdf')
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')